In [4]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.ml.classification import NaiveBayes # Removed RandomForestClassifier as it's not needed
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, Tokenizer, IDF, StringIndexer # Added StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [5]:
# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("BreastCancerPrediction")

# Stop any existing SparkContext if it's running
if 'SpContext' in globals() and SpContext:
    SpContext.stop()
SpContext = SparkContext(conf = conf)

# Create a SparkSession
SpSession = SparkSession.builder.getOrCreate()

print("Spark Context and Session initialized.")

Spark Context and Session initialized.


In [6]:
!wget -O 10-patient-characteristics.csv https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/10-patient-characteristics.csv

--2026-04-12 07:47:02--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/10-patient-characteristics.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 402502 (393K) [application/octet-stream]
Saving to: ‘10-patient-characteristics.csv’

10-patient-characte 100%[===================>] 393.07K  --.-KB/s    in 0.02s   

2026-04-12 07:47:02 (17.8 MB/s) - ‘10-patient-characteristics.csv’ saved [402502/402502]



In [7]:
print("== Inspecting 10-patient-characteristics.csv ==")

# Load the CSV file into an RDD
patient_data_rdd = SpContext.textFile("10-patient-characteristics.csv", 2)
patient_data_rdd.cache()

print(f"\nTotal instances loaded from patient characteristics file: {patient_data_rdd.count()}")
print("\nFirst 5 rows of raw patient characteristics data:")
for row in patient_data_rdd.take(5):
    print("\n" + row)

== Inspecting 10-patient-characteristics.csv ==

Total instances loaded from patient characteristics file: 4004

First 5 rows of raw patient characteristics data:

Addictive Behavior,Excluded: Patients whose use of alcohol or drugs is sufficient to impair compliance with protocol requirements.

Addictive Behavior,Current substance or alcohol use disorder as determined by the SCID or by positive drug toxicology results

Addictive Behavior,Subjects who consume >14 alcoholic drinks per week.

Addictive Behavior,Excessive consumption of xanthine-based beverages

Addictive Behavior,History of drug abuse or use of illegal drugs: use of soft drugs (marijuana  pot) within 3 months of the screening visit or hard drugs (cocaine  PCP  crack)within 1 year of the screening visit


In [8]:
print("== Start transforming patient characteristics data ==")
def TransformPatientCharacteristicsToVector(inputStr):
    attList = inputStr.split(",", 1) # Split only on the first comma to handle commas in the characteristic text
    # Assuming the first part is the label (e.g., 'age', 'gender') and the second is the free-text characteristic
    return [attList[0].strip(), attList[1].strip()] # label (string), characteristic content

patientXformed = patient_data_rdd.map(TransformPatientCharacteristicsToVector)

patientDf = SpSession.createDataFrame(patientXformed, ["label_string", "characteristic_text"])
patientDf.cache()
patientDf.select("label_string","characteristic_text").show(5, truncate=False) # truncate=False to see full text
print(f"\nTransformed DataFrame count: {patientDf.count()}")

== Start transforming patient characteristics data ==
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label_string      |characteristic_text                                                                                                                                                                             |
+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Addictive Behavior|Excluded: Patients whose use of alcohol or drugs is sufficient to impair compliance with protocol requirements.                                                                                 |
|Addictive Behavior|Current substance or alcohol use disorder as determined by the SCID or

In [9]:
print("== Start creating testing and training data ==")
# Split training and testing data (90% training, 10% testing)
(trainingData, testData) = patientDf.randomSplit([0.9, 0.1], seed=42)

print(f"\nTraining data count: {trainingData.count()}")
print(f"\nTest data count: {testData.count()}")

# Display first few rows of test data
print("\nFirst 5 rows of test data:")
testData.show(5, truncate=False)

== Start creating testing and training data ==

Training data count: 3628

Test data count: 376

First 5 rows of test data:
+------------------+----------------------------------------------------------------------------------------------+
|label_string      |characteristic_text                                                                           |
+------------------+----------------------------------------------------------------------------------------------+
|Addictive Behavior|Alcohol or drug abuse                                                                         |
|Addictive Behavior|Be current smokers of cigarettes (past 7 days)                                                |
|Addictive Behavior|Subject consumes more than 3 alcoholic beverages per day                                      |
|Addictive Behavior|Subjects who have a history of alcohol or drug abuse within 1 year of study entry             |
|Address           |Reside within zip codes corresponding to the

In [10]:
print("== Start building ML pipeline ==")

# Step 0: Index string labels to numerical labels
stringIndexer = StringIndexer(inputCol="label_string", outputCol="label")

# Step 1: Tokenize (break sentences into words)
tokenizer = Tokenizer(inputCol="characteristic_text", outputCol="words")

# Step 2: Calculate Term Frequency (TF)
hashingTF = HashingTF(inputCol=tokenizer.getOutputCol(), numFeatures=2000, outputCol="rawFeatures") # Increased features for better representation

# Step 3: Calculate TF-IDF (Term Frequency-Inverse Document Frequency)
tfIdf = IDF(inputCol=hashingTF.getOutputCol(), outputCol="features")

# Step 4: Naive Bayes Classifier
# NaiveBayes requires featuresCol and labelCol.
# Smooth parameter can be adjusted, default is 1.0 (Laplace smoothing)
nbClassifier = NaiveBayes(featuresCol="features", labelCol=stringIndexer.getOutputCol(), modelType="multinomial") # Use multinomial for text classification

# Build a pipeline, which consists of ML processing stages
pipeline = Pipeline(stages=[stringIndexer, tokenizer, hashingTF, tfIdf, nbClassifier])

print("\nML Pipeline constructed.")

== Start building ML pipeline ==

ML Pipeline constructed.


In [11]:
print("== Run ML pipeline ==")
# Train the model with the pipeline
nbPipeLineModel = pipeline.fit(trainingData) # Training phase (renamed model variable)

# Make predictions on the test data
prediction = nbPipeLineModel.transform(testData) # Testing phase

print("\n== Calculate and print evaluation ==")
# Evaluate accuracy
evaluator = MulticlassClassificationEvaluator(
    predictionCol="prediction",
    labelCol="label", # Label is now the indexed numerical label
    metricName="accuracy"
)
accuracyResult = evaluator.evaluate(prediction)

# Draw confusion matrix
print("\nConfusion Matrix:")
# Show actual string labels along with numerical labels if desired for better understanding
# For now, just show numerical labels
prediction.groupBy("label","prediction").count().show()

print(f"Accuracy: {accuracyResult:.4f}")
print(f"\nError: {1 - accuracyResult:.4f}")

# Stop Spark Context
SpContext.stop()
print("\nSpark Context stopped.")

== Run ML pipeline ==

== Calculate and print evaluation ==

Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|      12.0|    2|
|  7.0|       7.0|   11|
|  1.0|      16.0|    1|
|  0.0|      10.0|    1|
|  1.0|      17.0|    1|
| 14.0|       2.0|    1|
|  0.0|      16.0|    5|
|  1.0|      11.0|    1|
| 12.0|      12.0|    7|
|  1.0|      26.0|    1|
| 16.0|       2.0|    1|
|  1.0|       1.0|   21|
| 10.0|      10.0|    7|
|  0.0|      26.0|    5|
|  1.0|       6.0|    1|
|  0.0|       1.0|    6|
|  0.0|      15.0|    2|
|  0.0|      22.0|    2|
|  0.0|       9.0|    1|
|  7.0|       2.0|    1|
+-----+----------+-----+
only showing top 20 rows
Accuracy: 0.6755

Error: 0.3245

Spark Context stopped.
